# 02 — Recuperação documental e Recall

A recuperação é lexical: TF-IDF de palavras (70%) e de n-gramas de caracteres (30%), com similaridade de cosseno, até três trechos e limiar mínimo de 0,15. Não utiliza embeddings densos nem ChromaDB.

Recall@k = média da fração de fontes relevantes recuperadas entre as perguntas que possuem ao menos uma referência no gabarito. Uma pergunta com duas fontes relevantes contribui com 0,5 se só uma for recuperada.

**Não há chamadas à LLM neste notebook.**

In [1]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

In [2]:
from app.core import retrieve

conjuntos = [
    ("Original (desenvolvimento)", ROOT / "avaliacao/perguntas.json",
     ROOT / "avaliacao/recall_resultados.json"),
    ("Independente (avaliação)", ROOT / "avaliacao/perguntas_novas.json",
     ROOT / "avaliacao/rodadas/avaliacao_independente_e68b6315fca6/recall_resultados.json"),
]

def calc_recall(perguntas, k):
    validas = [q for q in perguntas if q["trechos_relevantes"]]
    if not validas:
        return None
    return sum(len(set(q["trechos_relevantes"]) &
                   {r["id"] for r in retrieve(q["pergunta"], k=k)}) /
               len(set(q["trechos_relevantes"])) for q in validas) / len(validas)

metricas = []
for nome, arquivo, salvo in conjuntos:
    if arquivo.exists():
        perguntas = json.loads(arquivo.read_text(encoding="utf-8"))
        n_rotuladas = sum(bool(q["trechos_relevantes"]) for q in perguntas)
        r1, r3 = calc_recall(perguntas, 1), calc_recall(perguntas, 3)
        origem = "recalculado"
    elif salvo.exists():
        registro = json.loads(salvo.read_text(encoding="utf-8"))
        r1, r3 = registro["recall_at_1"], registro["recall_at_3"]
        n_rotuladas = "—"
        origem = "registro de avaliação salvo"
    else:
        raise FileNotFoundError(f"Não encontrei gabarito nem resultados da avaliação: {nome}")
    metricas.append((nome, n_rotuladas, r1, r3, origem))

print(f"{'Conjunto':31} {'Com fontes':>10} {'Recall@1':>10} {'Recall@3':>10}   Origem")
for nome, n, r1, r3, origem in metricas:
    print(f"{nome:31} {str(n):>10} {r1:>10.2%} {r3:>10.2%}   {origem}")

Conjunto                        Com fontes   Recall@1   Recall@3   Origem
Original (desenvolvimento)              24     89.58%     97.92%   recalculado
Independente (avaliação)                12     75.00%     83.33%   recalculado


## Justificativa do limiar de relevância

O limiar `0,15` foi selecionado previamente a partir de dez consultas de calibração registradas em `avaliacao/calibracao.json`, sendo quatro consultas com fonte e seis sem fonte. O conjunto é pequeno e não garante que o limiar seja ideal para outros enunciados.

In [3]:
calibracao = json.loads((ROOT / "avaliacao/calibracao_resultados.json").read_text(encoding="utf-8"))
pares = calibracao["candidatos"]
positivos = [float(c["top1_score"]) for c in pares if c["tem_fonte"]]
negativos = [float(c["top1_score"]) for c in pares if not c["tem_fonte"]]
limiar = float(calibracao["limiar_selecionado"])
print(f"Consultas de calibração: {len(pares)} ({len(positivos)} com fonte, {len(negativos)} sem fonte)")
print(f"Menor score entre positivas: {min(positivos):.5f}")
print(f"Maior score entre negativas: {max(negativos):.5f}")
print(f"Limiar registrado: {limiar:.2f}")
print("Limiar separa os dois grupos da calibração:", max(negativos) < limiar <= min(positivos))

Consultas de calibração: 10 (4 com fonte, 6 sem fonte)
Menor score entre positivas: 0.16649
Maior score entre negativas: 0.14355
Limiar registrado: 0.15
Limiar separa os dois grupos da calibração: True


## Exemplo de recuperação e erros de busca

O código abaixo mostra trechos recuperados para uma pergunta do gabarito e indica, por conjunto, quais fontes esperadas ficaram fora do top-3. As listas são calculadas a partir do corpus presente na execução.

In [4]:
exemplo = json.loads((ROOT / "avaliacao/perguntas.json").read_text(encoding="utf-8"))[0]
print("Pergunta:", exemplo["pergunta"])
print("Fontes esperadas:", exemplo["trechos_relevantes"])
for item in retrieve(exemplo["pergunta"], k=3):
    print(f"  {item['id']} (similaridade: {item['score']:.5f}) — {item['titulo']}")

for nome, arquivo, _ in conjuntos:
    if not arquivo.exists():
        continue
    perguntas = json.loads(arquivo.read_text(encoding="utf-8"))
    falhas = []
    for q in perguntas:
        if not q["trechos_relevantes"]:
            continue
        obtidos = {item["id"] for item in retrieve(q["pergunta"], k=3)}
        faltam = sorted(set(q["trechos_relevantes"]) - obtidos)
        if faltam:
            falhas.append((q["id"], faltam))
    print(f"\n{nome}: {len(falhas)} perguntas sem recuperação completa no top-3")
    for id_questao, faltantes in falhas[:12]:
        print(f"  {id_questao} — fontes faltantes: {', '.join(faltantes)}")

Pergunta: Onde foi relatado um buraco na calçada?
Fontes esperadas: ['S001']
  S001 (similaridade: 0.49539) — Registro simulado OC001 — buraco na calçada

Original (desenvolvimento): 1 perguntas sem recuperação completa no top-3
  Q24 — fontes faltantes: G004

Independente (avaliação): 2 perguntas sem recuperação completa no top-3
  N05 — fontes faltantes: S008
  N12 — fontes faltantes: S005


In [5]:
# Visualização opcional: o notebook continua funcionando sem matplotlib.
try:
    import matplotlib.pyplot as plt
    rotulos = [linha[0].split(" (")[0] for linha in metricas]
    posicoes = list(range(len(metricas)))
    largura = 0.34
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.bar([x - largura / 2 for x in posicoes], [linha[2] * 100 for linha in metricas], largura, label="Recall@1")
    ax.bar([x + largura / 2 for x in posicoes], [linha[3] * 100 for linha in metricas], largura, label="Recall@3")
    ax.set_xticks(posicoes, rotulos)
    ax.set_ylabel("Recall (%)")
    ax.set_ylim(0, 105)
    ax.set_title("Recuperação de fontes esperadas")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Gráfico opcional não disponível; os valores numéricos acima são suficientes.")

Gráfico opcional não disponível; os valores numéricos acima são suficientes.


## Interpretação

O conjunto original alcançou Recall@1 de 89,58% e Recall@3 de 97,92%; no conjunto independente os valores observados foram 75,00% e 83,33%. A redução indica menor cobertura de trechos esperados para os enunciados novos. Recall não mede a correção factual do Qwen; ela foi avaliada separadamente no notebook 03.